# 🎙️ Qwen3-TTS - Text to Speech con IA

**Optimizado para Kaggle T4 GPU**

Generador de voz con Qwen3-TTS-1.7B.

---

## 🎯 Características:

- ✅ Generación de voz natural
- ✅ Compatible con GPU T4 de Kaggle
- ✅ Interfaz Gradio fácil de usar
- ✅ Correcciones de CUDA incluidas

---

In [ ]:
# @title 🔧 Configuración Inicial y Verificación GPU

import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# Verificar GPU
print("🔍 Verificando GPU...")
!nvidia-smi --query-gpu=name,memory.total,driver_version,cuda_version --format=csv
print()

# Verificar PyTorch y CUDA
try:
    import torch
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    
    if torch.cuda.is_available():
        print(f"CUDA version: {torch.version.cuda}")
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"Compute capability: {torch.cuda.get_device_capability(0)}")
        
        # Test CUDA
        try:
            x = torch.tensor([1.0, 2.0]).cuda()
            print("\n✅ CUDA funcionando correctamente")
        except Exception as e:
            print(f"\n⚠️ Error CUDA: {e}")
            print("📦 Reinstalando PyTorch con soporte CUDA correcto...")
            !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
            print("✅ PyTorch reinstalado. Reinicia el kernel si es necesario.")
    else:
        print("\n❌ CUDA no disponible. Activa GPU en Settings → Accelerator → GPU")
except ImportError:
    print("PyTorch no instalado, se instalará en el siguiente paso")

In [ ]:
# @title 📦 Instalar Dependencias

print("📦 Instalando dependencias...\n")

# Instalar dependencias compatibles con Kaggle T4
!pip install -q "transformers==4.57.3" "accelerate>=1.2.0" "soundfile" "gradio>=5.0.0" "huggingface_hub"
!pip install -q qwen-tts

print("\n✅ Dependencias instaladas\n")

# Verificar versiones
import torch
import transformers
print(f"📦 Versiones instaladas:")
print(f"  • PyTorch: {torch.__version__}")
print(f"  • Transformers: {transformers.__version__}")
print(f"  • CUDA disponible: {torch.cuda.is_available()}")

In [ ]:
# @title 🚀 Cargar Modelo y Lanzar Interfaz

import os
import gc
import torch
import gradio as gr
from qwen_tts import Qwen3TTSModel
import soundfile as sf
import tempfile

# Optimizaciones
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

print("🚀 Cargando modelo Qwen3-TTS-1.7B...")
print("⏳ Esto puede tardar unos minutos...\n")

try:
    # Cargar modelo
    model = Qwen3TTSModel.from_pretrained(
        "Qwen/Qwen3-TTS-1.7B",
        torch_dtype=torch.float16,
        device_map="auto"
    )
    
    print("\n✅ Modelo cargado correctamente")
    print(f"📍 Dispositivo: {model.device}")
    
except Exception as e:
    print(f"\n❌ Error cargando modelo: {e}")
    print("\n🔧 Solución de problemas:")
    print("1. Verifica que GPU está activada (Settings → Accelerator → GPU)")
    print("2. Reinicia el kernel y ejecuta de nuevo")
    print("3. Revisa los logs para errores específicos de CUDA")
    raise

# Función TTS
def generar_voz(texto, tipo_voz="default"):
    """Generar audio a partir de texto"""
    try:
        with torch.no_grad():
            audio = model.generate(texto)
        
        # Guardar archivo temporal
        temp_file = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        sf.write(temp_file.name, audio["audio"], audio["sampling_rate"])
        
        return temp_file.name
        
    except Exception as e:
        return f"Error: {str(e)}"

# Crear interfaz Gradio
with gr.Blocks(title="Qwen3-TTS Generator") as demo:
    gr.Markdown("# 🎙️ Generador de Voz - Qwen3-TTS")
    gr.Markdown("Convierte texto a voz natural con IA")
    
    with gr.Row():
        texto_input = gr.Textbox(
            label="Texto a convertir",
            placeholder="Escribe el texto aquí...",
            lines=5
        )
    
    with gr.Row():
        tipo_voz = gr.Dropdown(
            choices=["default", "male", "female"],
            value="default",
            label="Tipo de voz"
        )
        btn_generar = gr.Button("🎙️ Generar Audio", variant="primary")
    
    audio_output = gr.Audio(label="Audio generado")
    
    btn_generar.click(
        fn=generar_voz,
        inputs=[texto_input, tipo_voz],
        outputs=audio_output
    )

print("\n🎵 Interfaz lista. Lanzando Gradio...\n")
demo.launch(share=True, debug=True)

## 🔧 Utilidades

In [ ]:
# @title 🧹 Limpiar Memoria GPU

import gc
import torch

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    print("✅ Memoria GPU liberada")
    print()
    !nvidia-smi
else:
    print("❌ GPU no disponible")

In [ ]:
# @title 📊 Verificar Estado del Sistema

import torch
import psutil

print("📊 Estado del sistema:\n")
print(f"CPU: {psutil.cpu_percent()}% uso")
print(f"RAM: {psutil.virtual_memory().percent}% uso")
print(f"GPU disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU usada: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"Memoria GPU total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")